In [28]:
# Prove su pre processing e feature extraction

import mne
from mne.decoding import CSP
from mne_bids import BIDSPath, read_raw_bids
import numpy as np
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from pathlib import Path
import matplotlib.pyplot as plt


# Path per il dataset
root = "../data"
root = (Path(root).resolve())
runs = ["4", "8", "12"]   # Le run che vengono prese in considerazione
#runs = ["4"]   # Le run che vengono prese in considerazione

X_all = []  # Vettore X con tutti i dati di tutte le finestre
y_all = []  # Vettore y con tutte le etichette di tutte le finestre

for i in range(1, 4):
    subject = f"{i:03d}"
    # Per ogni soggetto eseguiamo la scansione sulle run di nostro interesse
    for run in runs:
        bids_path = BIDSPath(
            subject=subject,
            task="motion",
            run=run,
            datatype="eeg",
            root=root
        )


        raw = read_raw_bids(bids_path, verbose=False)
        events, event_id = mne.events_from_annotations(raw, verbose=False)
        channels = ["C3", "Cz", "C4"];
        # Plot raw

        # Fase 1: Preprocessing
        raw.load_data(verbose=False)
        raw.filter(l_freq=1, h_freq=40, verbose=False)
        # raw.plot(scalings='auto', show=True, block=True, verbose=False)
        # raw.set_eeg_reference('average', projection=False, verbose=False)

        # Fase 2: Feature Extraction
        epochs = mne.Epochs(
            raw,
            events,
            event_id=event_id,
            tmin=-1.0,
            tmax=3.0,
            baseline=(-1.0, 0.0),
            preload=True,
            verbose=False
        )
        y = epochs.events[:, 2]
            
        y_binary = np.where(y == 1, 0, 1) 

        psd = epochs.compute_psd(
            method="welch",
            fmin=1,
            fmax=45,
            n_fft=256,
            verbose=False
        )
        psd_data = psd.get_data()
        freqs = psd.freqs

        bands = {
            "delta": (1, 4),
            "theta": (4, 8),
            "alpha": (8, 13),
            "beta": (13, 30),   
        }

        band_features = []
        for band_limits in bands.values():
            fmin, fmax = band_limits
            band_mask = (freqs >= fmin) & (freqs < fmax)
            band_power = psd_data[:, :, band_mask].mean(axis=2)
            band_features.append(band_power)

        features = np.concatenate(band_features, axis=1)

        X_all.append(features)
        y_all.append(y_binary)
        print(f"Soggetto {subject} run {run} - Campioni: {features.shape[0]}, Feature per campione: {features.shape[1]}, Etichette: {y.shape[0]}")


Soggetto 001 run 4 - Campioni: 29, Feature per campione: 256, Etichette: 29
Soggetto 001 run 8 - Campioni: 29, Feature per campione: 256, Etichette: 29
Soggetto 001 run 12 - Campioni: 29, Feature per campione: 256, Etichette: 29
Soggetto 002 run 4 - Campioni: 29, Feature per campione: 256, Etichette: 29
Soggetto 002 run 8 - Campioni: 29, Feature per campione: 256, Etichette: 29
Soggetto 002 run 12 - Campioni: 29, Feature per campione: 256, Etichette: 29
Soggetto 003 run 4 - Campioni: 29, Feature per campione: 256, Etichette: 29
Soggetto 003 run 8 - Campioni: 29, Feature per campione: 256, Etichette: 29
Soggetto 003 run 12 - Campioni: 29, Feature per campione: 256, Etichette: 29


In [29]:
# pca = PCA(n_components=10, random_state=0)
# pca_features = pca.fit_transform(features)

# print("Feature matrix shape:", features.shape)
# print("PCA output shape:", pca_features.shape)
# # Stampa i vari parametri del pca
# comp = pca.components_[0]
# indices = np.argsort(np.abs(comp))[::-1]

# for i in indices[:20]:
#     print(i, comp[i])

# print(pca.explained_variance_ratio_)

# for i, ch in enumerate(raw.ch_names):
#     print(i, ch, " ", end = "")

# Concatenazione finale
X_all = np.vstack(X_all)
y_all = np.concatenate(y_all)

# Controllo distribuzione classi
unique, counts = np.unique(y_all, return_counts=True)
print("Distribuzione classi:", dict(zip(unique, counts)))

# Pipeline: scaling + LDA
clf = make_pipeline(
    StandardScaler(),
    PCA(n_components=10, random_state=0),
    SVC(kernel="linear", C=1.0, gamma='scale')  # robusto e stabile
)

# Cross-validation stratificata
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Valutazione con più metriche
scores = cross_validate(
    clf,
    X_all,
    y_all,
    cv=cv,
    scoring=["accuracy", "balanced_accuracy"],
    return_train_score=False
)

print("Accuracy per fold:", scores["test_accuracy"])
print("Accuracy media:", scores["test_accuracy"].mean())

print("Balanced accuracy per fold:", scores["test_balanced_accuracy"])
print("Balanced accuracy media:", scores["test_balanced_accuracy"].mean())


Distribuzione classi: {np.int64(0): np.int64(126), np.int64(1): np.int64(135)}
Accuracy per fold: [0.56603774 0.55769231 0.65384615 0.53846154 0.53846154]
Accuracy media: 0.5708998548621189
Balanced accuracy per fold: [0.56552707 0.55925926 0.65481481 0.54074074 0.54222222]
Balanced accuracy media: 0.5725128205128205
